https://stable-baselines3.readthedocs.io/en/master/

In [1]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" "autorom[accept-rom-license]" imageio imageio-ffmpeg
!AutoROM --accept-license


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\marce\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	C:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\AutoROM\roms

Existing ROMs will be overwritten.


In [2]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

env = gym.make("ALE/Freeway-v5")
obs, info = env.reset()

print("Ambiente OK!")
print("Observação:", obs.shape)
print("Ações:", env.action_space)

env.close()

Ambiente OK!
Observação: (210, 160, 3)
Ações: Discrete(3)


## Monitoramento com TensorBoard

Inicia o TensorBoard **antes** do treinamento. O painel lê a pasta `./freeway_tensorboard/` e atualiza sozinho enquanto o modelo treina (clique no botão de recarregar do painel se necessário).

Métricas principais:
- `rollout/ep_rew_mean` – recompensa média por episódio (quantas vezes a galinha atravessou)
- `rollout/ep_len_mean` – duração média dos episódios
- `train/loss`, `train/value_loss`, `train/policy_gradient_loss` – perdas do PPO
- `train/entropy_loss` – entropia da política (exploração)
- `train/approx_kl`, `train/clip_fraction` – tamanho das atualizações da política

In [3]:
import sys
import os

print("Python utilizado:", sys.executable)
print("Versão:", sys.version)

Python utilizado: C:\Users\marce\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
Versão: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [4]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "tensorboard"
])

0

In [5]:
import tensorboard

print("TensorBoard instalado:", tensorboard.__version__)

TensorBoard instalado: 2.21.0


In [6]:
import subprocess
import sys

logdir = "./freeway_tensorboard/"

process = subprocess.Popen([
    sys.executable,
    "-m",
    "tensorboard.main",
    "--logdir",
    logdir,
    "--port",
    "6006"
])

print("TensorBoard PID:", process.pid)
print("Acesse: http://localhost:6006")

TensorBoard PID: 25488
Acesse: http://localhost:6006


In [7]:
%pip install -q tensorboard

import os, sys, shutil

# Aponta o %tensorboard para o executável do mesmo Python do kernel
# (evita o erro "Could not find `tensorboard`" quando a pasta Scripts/bin não está no PATH)
base = os.path.dirname(sys.executable)
candidatos = [
    os.path.join(base, "tensorboard.exe"),             # venv no Windows
    os.path.join(base, "Scripts", "tensorboard.exe"),  # Python "global" no Windows
    os.path.join(base, "tensorboard"),                 # Linux/Mac/Colab
]
tb = next((c for c in candidatos if os.path.exists(c)), shutil.which("tensorboard"))
if tb:
    os.environ["TENSORBOARD_BINARY"] = tb
print("TensorBoard:", tb)

%load_ext tensorboard
%tensorboard --logdir ./freeway_tensorboard/

Note: you may need to restart the kernel to use updated packages.
TensorBoard: c:\Users\marce\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\Scripts\tensorboard.EXE



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\marce\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import tensorboard
print(tensorboard.__version__)

2.21.0


In [25]:

import gymnasium as gym
import ale_py

from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.monitor import Monitor

gym.register_envs(ale_py)


def make_env():
    env = gym.make("ALE/Freeway-v5")
    env = Monitor(env)
    return AtariWrapper(env)


env = make_env()

model = PPO(
    "CnnPolicy",
    env,
    verbose=1,

    # N_STEPS:
    # Quantidade de interações coletadas antes de cada
    # atualização da política (tamanho do rollout).
    #
    # Valor pequeno:
    # + Atualizações mais frequentes.
    # + Menor memória necessária para armazenar o rollout.
    # - Menos experiências disponíveis por atualização.
    # - Estimativas do gradiente podem apresentar maior variância.
    #
    # Valor grande:
    # + Mais experiências disponíveis por atualização.
    # + Pode proporcionar estimativas de gradiente mais estáveis.
    # - Atualizações menos frequentes.
    # - Maior consumo de memória.
    # - Pode aumentar o custo de cada iteração e reduzir
    #   a capacidade de adaptação a mudanças no ambiente.
    #
    # Atenção: não significa que valores maiores sejam sempre melhores.
    n_steps=2048,

    # BATCH_SIZE:
    # Número de amostras utilizadas em cada minibatch
    # durante a otimização da rede neural.
    batch_size=256,

    learning_rate=2.5e-4,

    # Diretório de armazenamento das métricas do TensorBoard.
    tensorboard_log="./freeway_tensorboard/"
)

model.learn(
    # TOTAL_TIMESTEPS:
    # Orçamento TOTAL de interações do agente com o ambiente.
    #
    # Valor pequeno:
    # + Treinamento mais rápido.
    # + Menor custo computacional.
    # - Menos experiências para aprender.
    # - Maior risco de não aprender uma política satisfatória.
    #
    # Valor grande:
    # + Mais oportunidades de aprendizado.
    # + Maior possibilidade de desenvolver uma política melhor.
    # - Maior tempo de treinamento.
    # - Maior consumo de recursos computacionais.
    #
    # Diferentemente de n_steps, controla a duração global
    # do treinamento, e não o tamanho de cada rollout.
    total_timesteps=50_000,

    tb_log_name="PPO_freeway",

    # Exibe a barra de progresso do treinamento.
    progress_bar=True
)

# Salva os parâmetros treinados.
model.save("freeway_ppo")

env.close()


Using cpu device

Wrapping the env in a DummyVecEnv.

Wrapping the env in a VecTransposeImage.

Logging to ./freeway_tensorboard/PPO_freeway_4

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.05e+03 |
|    ep_rew_mean     | 0.25     |
| time/              |          |
|    fps             | 486      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.05e+03    |
|    ep_rew_mean          | 0.375       |
| time/                   |             |
|    fps                  | 362         |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008401556 |
|    clip_fraction        | 0.0592      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.0481     |
|    learning_rate        | 0.00025     |
|    loss                 | 0.00347     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00281    |
|    value_loss           | 0.0945      |
-----------------------------------------

... (20 outputs left out)

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 2.05e+03      |
|    ep_rew_mean          | 16.2          |
| time/                   |               |
|    fps                  | 249           |
|    iterations           | 23            |
|    time_elapsed         | 188           |
|    total_timesteps      | 47104         |
| train/                  |               |
|    approx_kl            | 0.00017730566 |
|    clip_fraction        | 0.00396       |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.153        |
|    explained_variance   | 0.455         |
|    learning_rate        | 0.00025       |
|    loss                 | 0.0707        |
|    n_updates            | 220           |
|    policy_gradient_loss | -0.000259     |
|    value_loss           | 0.191         |
-------------------------------------------

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 2.05e+03     |
|    ep_rew_mean          | 16.4         |
| time/                   |              |
|    fps                  | 247          |
|    iterations           | 24           |
|    time_elapsed         | 198          |
|    total_timesteps      | 49152        |
| train/                  |              |
|    approx_kl            | 0.0010583979 |
|    clip_fraction        | 0.00474      |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.145       |
|    explained_variance   | 0.501        |
|    learning_rate        | 0.00025      |
|    loss                 | 0.0846       |
|    n_updates            | 230          |
|    policy_gradient_loss | 4.91e-05     |
|    value_loss           | 0.184        |
------------------------------------------

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 2.05e+03      |
|    ep_rew_mean          | 16.6          |
| time/                   |               |
|    fps                  | 246           |
|    iterations           | 25            |
|    time_elapsed         | 207           |
|    total_timesteps      | 51200         |
| train/                  |               |
|    approx_kl            | 0.00028738228 |
|    clip_fraction        | 0.00391       |
|    clip_range           | 0.2           |
|    entropy_loss         | -0.129        |
|    explained_variance   | 0.5           |
|    learning_rate        | 0.00025       |
|    loss                 | 0.106         |
|    n_updates            | 240           |
|    policy_gradient_loss | -0.000371     |
|    value_loss           | 0.228         |
-------------------------------------------

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51,200/50,000  [ 0:03:24 < 0:00:00 , 218 it/s ]

VIDEO DA GALINHA

In [21]:
import gymnasium as gym
import ale_py
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import AtariWrapper
from IPython.display import display, Video
import imageio

# Registrar os ambientes Atari
gym.register_envs(ale_py)

# Criar o ambiente
env = AtariWrapper(
    gym.make("ALE/Freeway-v5", render_mode="rgb_array")
)

# Carregar o modelo treinado
model = PPO.load("freeway_ppo", env=env)

print("Modelo carregado com sucesso!")

# Executar o agente e gravar um vídeo
obs, info = env.reset()
frames = []

for step in range(5000):
    frames.append(env.render())

    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs, info = env.reset()

env.close()

# Salvar e exibir o vídeo
imageio.mimsave("freeway_trained.mp4", frames, fps=30)
display(Video("freeway_trained.mp4", embed=True))

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
Modelo carregado com sucesso!


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Carrega o modelo e salva o video